In [ ]:
import csv
import sys
import numpy as np
import os
csv.field_size_limit(sys.maxsize)
import ast
import random
import matplotlib.pyplot as plt
import time
import pickle
import gc
from gpcam import GPOptimizer
from datetime import datetime
import shutil
from functools import partial
import dask
from dask.distributed import Client
import os
import time
from gpAMRutils import *

## Tuning options to consider
- Filtering data by block or as a function of the global data?
- What fraction of candidates should be used, > mean SD? top 10%, top n candidates?
- How will data noise be estimated and set?
- RAM: submitting ask several times means that $\kappa \in \mathbb{R}^{12000 \times 12000}$ is possibly stored many times at once. At the same time the interpolator in the kernel stores temporarily 2(3) matrices of shape (len(x_data)^2)

## SETTINGS

README:
- The density example grid is 64 by 64
- Each block is 16 x 16, making for 4 by 4 blocks
- We only need 4 GPUs/CPUs, therefore "./allocateCPUs.sh 4 16"
- And then start the dask scheduler and workers: "./launch-dask-moduleCPU.sh 4 16"

In [ ]:
#HPC SETTINGS:
number_of_workers = 16 #match dask launch script and allocation (allocate_CPUs.sh 4 16), and no block x * y
number_of_threads = 32 #currently not used


#DATA CURATION
normalizing = True #y_data in [0,1]
filtering = False #remove data below a threshold
tol_ratio = 1e-2 #remove data < tol_ratio * max(data)

#REFINEMENT LEVEL
refinement_res = 2 #refinement level


#PLOTTING
plotting = True #plot posterior mean and var
plot_resx = 100
plot_resy =  100
plot_every = 2 #iterations


#BLOCKING
no_blocks_x = 4
no_blocks_y = 4
pad = 10 #>= 1

#FILE HANDLING
filename = "plot.nx64.2d.AMRLevel.hdf5"
index = "density"
chombo_path = "./ChomboOut/"
gpcam_path = "./gpCAMOut/"
rename_file = True #if True, Chombo file will be read and then renamed for bookkeeping; if used, turn delete_file = False
delete_file = True #if true, rename_file is ignored

#GP setup
percentile = 50 ##of all candidates considered, return the top percentile
uncertainty_cutoff = 0.01
init_hyperparameters = np.array([0.000001, 10., 10., 0.]) #signal SD(!), length scale baseline, gradient sensitivity
hyperparameter_bounds = np.array([[-4., 4.],
                                  [1., 50.],
                                  [1., 50.],
                                  [-10., 10.]
                                 ])
match_hps = True
noise = 0.001 #variance #we normalize data to [0,1] so this is about 3% SD 


#####TEST MODE###################################
#################################################
#filename = "Testdensity64.hdf5"
#rename_file = False #if True, Chombo file will be read and then renamed for bookkeeping; if used, turn delete_file = False
#delete_file = False #if True, rename_file is ignored
#################################################




In [ ]:
#check if Chombo file already exists
if os.path.exists(chombo_path+"ready.txt") and os.path.exists(chombo_path+filename):
    print("The data file ", chombo_path+filename," already exists in the repo. Should I delete it? y/n")
    dec = input()
    if dec  == "y":
            print("File removed ...")
            os.remove(chombo_path+filename)
            os.remove(chombo_path+"ready.txt")
    elif dec == "n": print("file not deleted")
    else: print("This was not a viable option, run the cell again!")

In [ ]:
# ###TEST DATA AND CHECK FORMATTING
# data = read_fileIII(chombo_path, filename, index, rename = rename_file, delete=delete_file, n_sub_x = no_blocks_x, n_sub_y = no_blocks_y, pad_x=pad, pad_y=pad)

# if normalizing:
#      print("normalizing")
#      data = normalize_data(data)
# print(data)
# if filtering: data = filter_all_data(data, tol_ratio)
# print("global data")
# plt.scatter(data["global_coordinates"][:,0], data["global_coordinates"][:,1],c = data["global_funcvalues"])
# plt.show()
# for n in data["local_data"]:
#      print("local data", n)
#      plt.scatter(data["local_data"][n]["points"][:,0], data["local_data"][n]["points"][:,1], c = data["local_data"][n]["values"])
#      plt.colorbar()
#      plt.show()


# refinement_candidates = make_refinement_candidates(data, refinement_level=refinement_res)
# global_refinement_grid = refinement_candidates["global_candidates"]
# local_refinement_grids = refinement_candidates["local_candidates"]
# print("global_refinement_grid")
# plt.scatter(global_refinement_grid[:,0], global_refinement_grid[:,1], s = 0.1)
# plt.show()
# for n in data["local_data"]:
#     print("local refinement grid ", n)
#     plt.scatter(local_refinement_grids[n][:,0], local_refinement_grids[n][:,1], s = 0.1)
# plt.show()

## Dask Client

In [ ]:
scheduler_file = os.path.join(os.environ["SCRATCH"], "scheduler_filegpAMR.json")
dask.config.config["distributed"]["dashboard"]["link"] = "{JUPYTERHUB_SERVICE_PREFIX}proxy/{host}:{port}/status" 

client = init_client(scheduler_file, number_of_workers)

In [ ]:
client

## GP set-up

In [ ]:
from scipy.interpolate import griddata
from gpcam.kernels import *
from scipy.interpolate import RBFInterpolator
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import spsolve
from scipy.spatial.distance import cdist

# Wendland C^2 compactly supported kernel (support radius = 1)
def wendland_c2(r):
    out = np.zeros_like(r)
    mask = r < 1.0
    rm = 1 - r[mask]
    out[mask] = rm**4 * (4*r[mask] + 1)
    return out


# Derivative of Wendland C^2 w.r.t. r
def wendland_c2_prime(r):
    out = np.zeros_like(r)
    mask = r < 1.0
    rm = 1 - r[mask]
    # φ'(r) = d/dr [ (1-r)^4 * (4r +1) ]
    #       = -4*(1-r)^3*(4r+1) + (1-r)^4 * 4
    out[mask] = -4 * rm**3 * (4*r[mask] + 1) + 4 * rm**4
    return out

from scipy.spatial import cKDTree
from scipy.sparse import csr_matrix, identity


# class WendlandRBF:
#     def __init__(self, x, y, epsilon=0.5, reg=1e-10):
#         self.x = np.asarray(x, dtype=float)
#         self.y = np.asarray(y, dtype=float)
#         self.epsilon = float(epsilon)
#         self.support = 1.0 / self.epsilon          # support in *actual* distance
#         self.tree = cKDTree(self.x)
#         N = self.x.shape[0]

#         # build ONLY in-support entries -> genuinely sparse, no O(N^2) peak
#         D = self.tree.sparse_distance_matrix(self.tree, self.support,
#                                              output_type="coo_matrix")
#         vals = wendland_c2(D.data * self.epsilon)   # diag dist=0 -> phi(0)=1
#         K = csr_matrix((vals, (D.row, D.col)), shape=(N, N))
#         K.eliminate_zeros()
#         K = K + reg * identity(N, format="csr")
#         self.w = spsolve(K, self.y)                 # SPD; cholmod is ~2x if available

#     def __call__(self, xnew):
#         xnew = np.atleast_2d(np.asarray(xnew, dtype=float))
#         Dq = cKDTree(xnew).sparse_distance_matrix(self.tree, self.support,
#                                                   output_type="coo_matrix")
#         vals = wendland_c2(Dq.data * self.epsilon)
#         K = csr_matrix((vals, (Dq.row, Dq.col)),
#                        shape=(xnew.shape[0], self.x.shape[0]))
#         return np.asarray(K @ self.w)

#     def gradient(self, xnew):
#         """Return |grad s| at each query point. Shape (M,)."""
#         xnew = np.atleast_2d(np.asarray(xnew, dtype=float))
#         M, d = xnew.shape
#         nbrs = self.tree.query_ball_point(xnew, r=self.support)   # list of arrays

#         counts = [len(p) for p in nbrs]
#         if sum(counts) == 0:
#             return np.zeros(M)
#         qidx = np.repeat(np.arange(M), counts)
#         didx = np.concatenate(nbrs).astype(int)

#         diffs = xnew[qidx] - self.x[didx]            # (P, d)
#         u = np.linalg.norm(diffs, axis=1)            # unscaled distance
#         r = u * self.epsilon
#         m = (u > 0) & (r < 1.0)
#         qidx, didx, diffs, u, r = qidx[m], didx[m], diffs[m], u[m], r[m]

#         # CORRECT chain rule: one epsilon, divide by UNSCALED u
#         coeff = self.w[didx] * wendland_c2_prime(r) * self.epsilon / u
#         grad = np.zeros((M, d))
#         np.add.at(grad, qidx, coeff[:, None] * diffs)
#         return np.linalg.norm(grad, axis=1)

from scipy.interpolate import CloughTocher2DInterpolator, NearestNDInterpolator
def padded_ct(points, values, pad_frac=0.1, n_ghost=60):
    lo, hi = points.min(0), points.max(0)
    span = hi - lo
    plo, phi = lo - pad_frac*span, hi + pad_frac*span
    t = np.linspace(0, 1, n_ghost)
    ring = np.vstack([
        np.c_[plo[0] + t*(phi[0]-plo[0]), np.full_like(t, plo[1])],
        np.c_[plo[0] + t*(phi[0]-plo[0]), np.full_like(t, phi[1])],
        np.c_[np.full_like(t, plo[0]), plo[1] + t*(phi[1]-plo[1])],
        np.c_[np.full_like(t, phi[0]), plo[1] + t*(phi[1]-plo[1])],
    ])
    ghost = NearestNDInterpolator(points, values)(ring)   # Neumann-style extension
    P = np.vstack([points, ring])
    V = np.concatenate([values, ghost])
    return CloughTocher2DInterpolator(P, V, fill_value=0.), span

def int_obj(x_data, y_data):
    rbf, span = padded_ct(x_data, y_data)
    return  rbf, span

def interpolator(rbf, x):
    return rbf(x)

def interpolator_grad(rbf, span, x):
    h = 1e-3 * span.min()
    ex, ey = np.array([h, 0]), np.array([0, h])
    gx = (rbf(x + ex) - rbf(x - ex)) / (2*h)
    gy = (rbf(x + ey) - rbf(x - ey)) / (2*h)
    norm = np.hypot(gx, gy)
    return norm, gx, gy


def kernelPDEII(x1, x2, hps, args):
    """
    Non-stationary Gibbs kernel (gpAMR Eq. 4) with the length-scale field
    conditioned on the PDE-solution gradient (Eq. 5):

        sigma_f(x) = |q(x)|                          # signal std  <- solution VALUE
        ell(x)     = ell0 / (1 + beta |grad q(x)|)   # length scale <- solution GRADIENT

    hps[0] : signal-variance scale coefficient
    hps[1] : ell0, the smooth-region (maximum) length scale
    hps[2] : beta, gradient sensitivity of the length scale  (NEW)
    """
    x_data = args["x_data"]
    y_data = args["y_data"]
    D = x1.shape[1]
    rbf = args["rbf"]
    span = args["span"]

    # signal variance from the solution VALUE: sigma_f(x) = |q(x)|
    #sf1 = np.abs(rbf(x1))
    #sf2 = np.abs(rbf(x2))
    #rbf, span = int_obj(x_data, y_data)
    sf1 = abs(interpolator(rbf, x1))
    sf2 = abs(interpolator(rbf, x2))

    # length scale from the solution GRADIENT: small where |grad q| is large
    #g1 = np.asarray(rbf.gradient(x1))
    #g2 = np.asarray(rbf.gradient(x2))
    g1, gx1, gy1 = interpolator_grad(rbf, span, x1)
    g2, gx2, gy2 = interpolator_grad(rbf, span, x2)
    
    gmag1 = np.abs(g1) if g1.ndim == 1 else np.linalg.norm(g1, axis=1)
    gmag2 = np.abs(g2) if g2.ndim == 1 else np.linalg.norm(g2, axis=1)
    del rbf

    ell0, beta = hps[1], hps[2]**2
    l1 = ell0 / (1.0 + beta * gmag1)     # contracts where the field is steep (beta 0 --> no gradient sensitivity)
    l2 = ell0 / (1.0 + beta * gmag2)

    # Gibbs non-stationary SE kernel (Eq. 4, isotropic ell)
    denom = (l1**2)[:, None] + (l2**2)[None, :]          # ell(x1)^2 + ell(x2)^2
    prefactor = (2.0 * np.outer(l1, l2) / denom) ** (D / 2.0)
    d2 = get_distance_matrix(x1, x2) ** 2
    gibbs = prefactor * np.exp(-d2 / denom)
    return hps[0]**2 * gibbs

def meanf(x,hps, args):
    m = args["block_mean"]
    #print(m)
    return np.zeros(len(x)) + m

def acq_func(x, gp):
    x = np.asarray(x)
    return np.sqrt(gp.posterior_covariance(x, variance_only=True)["v(x)"])

In [ ]:
import numpy as np


def kernelPDEII(x1, x2, hps, args):
    """
    Anisotropic non-stationary Gibbs kernel (gpAMR Eq. 4), directional extension.

    Each axis gets its own length-scale field that contracts based on the PDE
    solution's gradient COMPONENT along that axis (dq/dx_d), NOT the gradient
    norm |grad q|. Using the norm makes both axes contract identically
    (isotropic), which defeats the directional intent. The gradient sensitivity
    `beta` is shared across directions:

        ell_d(x) = ell0_d / (1 + beta * |dq/dx_d|)              d = 0 .. D-1

        k(x, x') = sf^2
                   * PROD_d [ 2 * l1_d * l2_d / (l1_d^2 + l2_d^2) ]^(1/2)   # prefactor
                   * exp( - SUM_d (x_d - x'_d)^2 / (l1_d^2 + l2_d^2) )      # anisotropic SE

    This is the diagonal Paciorek-Schervish construction. It is guaranteed
    symmetric positive semidefinite for ANY per-axis field ell_d(x) > 0, which
    the denominator (1 + beta*|g_d|) >= 1 with ell0_d > 0, beta >= 0 enforces.
    In the zero-gradient limit it reduces to a standard anisotropic SE kernel;
    setting ell0_x = ell0_y and swapping components for the norm recovers the
    original isotropic kernel.

    Hyperparameters (D = 2 case shown; layout generalizes to any D):
        hps[0] : sf       signal-std scale       -> kernel returns sf^2 * (...)
        hps[1] : ell0_x   max length scale along axis 0 (flat-region corr. length)
        hps[2] : ell0_y   max length scale along axis 1 (flat-region corr. length)
        hps[3] : b        raw gradient sensitivity; effective beta = b^2 >= 0,
                          SHARED across directions (b^2 keeps it non-negative)

    For general D: hps[1 : 1+D] are the per-axis ell0_d, hps[1+D] is b.

    Recommended training bounds  (grid/index units: domain [0,64]^2, dx ~ 1)
    ----------------------------------------------------------------------------
        hps[0] : [1e-2, 1e1]    data-dependent; ~ sqrt(mean-square of y) is the
                                center of mass. Widen if y is not O(1).
        hps[1] : [2.0, 64.0]    >=2 cells (resolution floor) .. domain extent.
        hps[2] : [2.0, 64.0]    same. Search these in LOG space if fvGP allows;
                                length scales are multiplicative.
        hps[3] : [0.0, 2.2]     effective beta in [0, ~5]. 0.0 is a legitimate
                                stationary fallback the optimizer can rest on.
                                At |grad q| ~ 1 the effective beta reads directly
                                as the contraction factor (beta=1 halves ell).

    As a ready-to-use box:
        hps_bounds = np.array([[1e-2, 1e1],
                               [2.0, 64.0],
                               [2.0, 64.0],
                               [0.0, 2.2]])

    Conditioning note
    -----------------
    On a dense grid the not-PD / Cholesky failures come from the SMOOTH regime
    (large ell0 -> long correlations -> near-duplicate rows -> rank-deficient K),
    NOT from steep/contracted regions (those are well conditioned). Add a small
    diagonal jitter ~ 1e-6 * sf^2 to K; that is what protects the large-ell0
    corner. If you later need beta > ~5 with a hard floor on ell, switch to the
    floored field  ell_d = ell_min + (ell0_d - ell_min) / (1 + beta*|g_d|).
    """
    D = x1.shape[1]
    rbf = args["rbf"]
    span = args["span"]

    # --- gradient COMPONENTS at each point ---
    # interpolator_grad returns (|grad q|, dq/dx, dq/dy). We must use the
    # SEPARATE components, NOT the magnitude: the magnitude drives both axes
    # identically (isotropic contraction). Column order here MUST match the
    # column order of x1/x2 -- i.e. x1[:, 0] is the axis gx differentiates.
    _, gx1, gy1 = interpolator_grad(rbf, span, x1)
    _, gx2, gy2 = interpolator_grad(rbf, span, x2)
    del rbf
    gc1 = np.abs(np.stack([gx1, gy1], axis=1))        # (N1, 2): [|dq/dx|, |dq/dy|]
    gc2 = np.abs(np.stack([gx2, gy2], axis=1))        # (N2, 2)

    #FLOORED VERSION
    # hps[0]=sf, hps[1]=ell0_x, hps[2]=ell0_y, hps[3]=ell_min, hps[4]=sqrt(beta)
    # ell0    = np.asarray(hps[1:1 + D], dtype=float)   # (D,) per-axis MAX length scale
    # ell_min = 0.1                              # shared floor (> 0, < min ell0_d)
    # beta    = hps[1 + D] ** 2

    # l1 = ell_min + (ell0 - ell_min)[None, :] / (1.0 + beta * gc1)   # (N1, D)
    # l2 = ell_min + (ell0 - ell_min)[None, :] / (1.0 + beta * gc2)   # (N2, D)

    
    # --- per-direction length-scale fields (shared beta, forced non-negative) ---
    ell0 = np.asarray(hps[1:1 + D], dtype=float)       # (D,) = [ell0_x, ell0_y, ...]
    beta = hps[1 + D] ** 2                             # >= 0, shared across axes
    l1 = ell0[None, :] / (1.0 + beta * gc1)            # (N1, D)  contracts per axis
    l2 = ell0[None, :] / (1.0 + beta * gc2)            # (N2, D)

    # --- diagonal Gibbs kernel: independent product over dimensions ---
    prefactor = np.ones((x1.shape[0], x2.shape[0]))
    exponent = np.zeros_like(prefactor)
    for d in range(D):
        a = l1[:, d][:, None]                          # (N1, 1)
        b = l2[:, d][None, :]                          # (1, N2)
        denom_d = a ** 2 + b ** 2                      # ell_d(x1)^2 + ell_d(x2)^2
        dx_d = x1[:, d][:, None] - x2[:, d][None, :]   # per-axis coordinate diff
        prefactor *= np.sqrt(2.0 * a * b / denom_d)    # PSD-preserving amplitude
        exponent += -(dx_d ** 2) / denom_d
    gibbs = prefactor * np.exp(exponent)

    return hps[0] ** 2 * gibbs

In [ ]:
# init_hyperparameters = np.array([0.000001, 10., 10., 1000.]) #signal SD(!), length scale, gradient sensitivity


# data = read_fileIII(chombo_path, filename, index, rename = rename_file, delete=delete_file, n_sub_x = no_blocks_x, n_sub_y = no_blocks_y, pad_x=pad, pad_y=pad)
# x_data = data["global_coordinates"]

# _, gx1, gy1 = interpolator_grad(rbf, span, x_data)

# gc1 = np.abs(np.stack([gx1, gy1], axis=1))        # (N1, 2): [|dq/dx|, |dq/dy|]


# # --- per-direction length-scale fields (shared beta, forced non-negative) ---
# ell0 = np.asarray(init_hyperparameters[1:1 + 2], dtype=float)       # (D,) = [ell0_x, ell0_y, ...]
# beta = init_hyperparameters[-1]
# l1 = ell0[None, :] / (1.0 + beta * gc1)            # (N1, D)  contracts per axis
# plt.imshow(l1[:,1].reshape(64,64).T)
# plt.colorbar()

In [ ]:
data = read_fileIII(chombo_path, filename, index, rename = rename_file, delete=delete_file, n_sub_x = no_blocks_x, n_sub_y = no_blocks_y, pad_x=pad, pad_y=pad)
if normalizing: data = normalize_data(data)
if filtering: data = filter_all_data(data, tol_ratio)


refinement_candidates = make_refinement_candidates(data, refinement_level=refinement_res)
global_refinement_grid = refinement_candidates["global_candidates"]
local_refinement_grids = refinement_candidates["local_candidates"]


rbf, span = int_obj(data["global_coordinates"], data["global_funcvalues"])
GPs = {}
candidate_pools = {}
block_domains = {}
kernels = {}
LL = {}
print("Initializing GPs")
for blockID in data["local_data"]:
    kernels[blockID] = kernelPDEII
    x_data = data["local_data"][blockID]["points"]
    y_data = data["local_data"][blockID]["values"]
    #print("min/max y: ", np.min(y_data),np.max(y_data))
    block_init_hps = init_hyperparameters.copy()
    block_init_hps[0] = np.std(y_data) #gets squared in the kernel to get signal variance
    print("GP ", blockID," has hyperparameters ",  block_init_hps.round(2), "and signal min/max: ", np.min(y_data).round(2), np.max(y_data).round(2))
    GPs[blockID] = GPOptimizer(x_data, y_data, noise_variances=np.ones(y_data.shape) * noise,
                          kernel_function=kernels[blockID],
                          prior_mean_function=meanf,
                          init_hyperparameters = block_init_hps,
                          args={"active": True, "x_data": data["global_coordinates"], "y_data": data["global_funcvalues"], "rbf": rbf, "span": span, "block_mean": np.mean(y_data)}, linalg_mode = "CholInv")

    #define refinement res
    xmin = data["local_data"][blockID]["bounds"]["interior_cols"][0]
    xmax = data["local_data"][blockID]["bounds"]["interior_cols"][1]
    ymin = data["local_data"][blockID]["bounds"]["interior_rows"][0]
    ymax = data["local_data"][blockID]["bounds"]["interior_rows"][1]    
    candidate_pools[blockID] = [array for array in local_refinement_grids[blockID]]
    if not valid(candidate_pools[blockID]): raise Exception("Invalid candidate pool in blockID ", blockID)
    block_domains[blockID] = np.array([[xmin,xmax],
                                       [ymin,ymax]])
    #print("blockdomains:", block_domains[blockID])
print("Initialization done!")

all_candidates = global_refinement_grid
if not valid(all_candidates): raise Exception("Duplicates in global candidate pool")


GPs = client.scatter(GPs, broadcast=False, direct=True)
print("Initial training...")
train_futures = []
for blockID in data["local_data"]: 
    train_futures.append(train(client, hyperparameter_bounds, GPs[blockID],  max_iter = 1000, method = "mcmc"))
client.gather(train_futures)
print("Initial training done!")
for entry in GPs:
    LL[entry] = GPs[entry].result().log_likelihood()
    print("hyperparameters GP", entry," : ",GPs[entry].result().hyperparameters.round(2), "@ likelihood: ", LL[entry])
if match_hps:
    maxLL = max(LL, key=lambda k: LL[k])
    for entry in GPs: set_hps(client, GPs[entry], GPs[maxLL].result().hyperparameters)

###################################################
###################################################
###################################################
###################################################
###################################################
###################################################
###################################################
iteration_counter = 0
print("#######################")
print("start gpAMR iteration: ")
print("#######################")
suggestion_history = []
plot_counter = plot_every-1
training_at = [2,5,10,20,30,40,50,60,70,80,90,100,120,140,150,200]
while True:
    iteration_counter += 1
    print("")
    print("")
    print("++++++++++++++++++++++++++++++++++")
    print("start gpAMR iteration: ", iteration_counter)
    print("++++++++++++++++++++++++++++++++++")
    
    res = []
    #ASKING FOR SUGGESTIONS
    print("Asking for new suggestions")
    for blockID in data["local_data"]:
        if not GPs[blockID].result().args["active"]: continue
        print("        Asking GP ",blockID, " with ",len(GPs[blockID].result().x_data)," data points, for suggestions")
        candidate_pool = candidate_pools[blockID]
        #candidates = list(chunks(candidate_pool, number_of_threads))
        print("        ask for new suggestions.... Candidates  considered: ", len(candidate_pool))
        #for chunk in candidates: res.append(ask(client, chunk, GPs[ID], len(chunk), acq_func)) ###FAST BUT RAM INEFFICIENT
        #print(np.asarray(candidate_pool))
        res.append(ask(client, candidate_pool, GPs[blockID], len(candidate_pool), acq_func)) ###SLOWER BUT RAM EFFICIENT
    new = client.gather(res)

    print("    All GP agents reported suggestions...concatenating")
    SD = np.concatenate([x["f_a(x)"] for x in new])
    new = np.vstack([x["x"] for x in new])
    sorted_indices = np.argsort(SD)[::-1]
    SD = SD[sorted_indices]
    new = new[sorted_indices]
    uncertainty_tol = np.percentile(SD, percentile)
    plt.plot(SD)
    print("uncertainty tol: ", uncertainty_tol, "uncertainty cutoff: ", uncertainty_cutoff)
    plt.show()
    non_zero_ind = np.where(SD > max(uncertainty_tol,uncertainty_cutoff))
    if not valid(new): raise Exception("Non-Unique suggestions")
    suggestions = new[non_zero_ind]
    print("len(suggestions): ", len(suggestions))
    print("    Suggestions calculated, len:", len(suggestions))
    print("    Write suggestions for Chombo iteration... ", iteration_counter)
    #################################
    #suggestions = np.round(suggestions)
    #################################
    if not valid(suggestions): raise Exception("Non-Unique suggestions")
    write_file(gpcam_path, chombo_path, suggestions) ##send to data generator (simulation)
    print("    Suggestions written!")
    suggestion_history.append(suggestions)
    

    #PLOTTING
    plot_counter+=1
    if plotting and plot_counter==plot_every:
        plot_counter=0
        print("Generating plots...")
        mean_futures = []
        cov_futures = []
        for blockID in data["local_data"]:
            if not GPs[blockID].result().args["active"]: continue 
            x_plot = GPOptimizer.make_2d_x_pred(block_domains[blockID][0],block_domains[blockID][1],resx=plot_resx,resy=plot_resy)
            mean_futures.append(posterior_mean(client, x_plot,GPs[blockID]))
            cov_futures.append(posterior_covariance(client, x_plot,GPs[blockID]))
        means_tmp = client.gather(mean_futures)
        means = np.concatenate([mean["m(x)"] for mean in means_tmp])
        cov_tmp = client.gather(cov_futures)
        stds = np.concatenate([np.sqrt(cov["v(x)"]) for cov in cov_tmp])
        x_pred = np.vstack([mean["x_pred"] for mean in means_tmp])
        
        print("global data and suggestions:")
        plt.figure(figsize=(20,5))
        a = plt.scatter(data["global_coordinates"][:,0], data["global_coordinates"][:,1], c=data["global_funcvalues"], alpha=.2)
        plt.scatter(suggestions[:,0], suggestions[:,1], s=0.1, c='black', alpha=1.)
        plt.colorbar(a)
        plt.show()

        print("global mean and suggestions:")
        plt.figure(figsize=(20,5))
        plot2d(x_pred[:,0], x_pred[:,1], means, suggestions=suggestions, title= "mean and suggestions",filename=gpcam_path+"mean" + str(iteration_counter).zfill(4))
        
        print("global std and suggestions:")
        plot2d(x_pred[:,0], x_pred[:,1], stds, suggestions=suggestions, title= "std and suggestions", filename=gpcam_path+"std" + str(iteration_counter).zfill(4))

        ##write image to disc
        print("Write to file...")
        print("mean + suggestions")
        plt.figure(figsize=(20,5))
        a = plt.scatter(x_pred[:,0],x_pred[:,1],c = means, alpha=1.)
        plt.scatter(suggestions[:,0], suggestions[:,1], s=0.1, c='black', alpha=1.0)
        plt.xlim(data["domain"][0,0], data["domain"][0,1])
        plt.ylim(data["domain"][1,0], data["domain"][1,1])
        plt.colorbar(a)
        plt.savefig(gpcam_path+"mean_sugg" + str(iteration_counter).zfill(4))
        plt.show()
        ##write image to disc
        print("sd + suggestions")
        plt.figure(figsize=(20,5))
        a = plt.scatter(x_pred[:,0],x_pred[:,1], c = stds, alpha=0.5)
        plt.scatter(suggestions[:,0], suggestions[:,1], s=0.1, c='black', alpha=0.3)
        plt.xlim(data["domain"][0,0], data["domain"][0,1])
        plt.ylim(data["domain"][1,0], data["domain"][1,1])
        plt.colorbar(a)
        plt.savefig(gpcam_path+"std_sugg" + str(iteration_counter).zfill(4))
        plt.show()
        print("Done plotting for this iteration!")
    
    

    print("Reading Chombo file. Iteration: ", iteration_counter)
    data = read_fileIII(chombo_path, filename, index, rename = rename_file, delete=delete_file, n_sub_x = no_blocks_x, n_sub_y = no_blocks_y, pad_x=pad, pad_y=pad)
    rbf, span = int_obj(data["global_coordinates"], data["global_funcvalues"])
    if normalizing: data = normalize_data(data)
    if filtering: data = filter_all_data(data, tol_ratio)
    print("global dataset size: ", len(data["global_coordinates"]))
    print("filter tol: ", tol_ratio * np.max(data["global_funcvalues"]))
    print("Received data:")
    plt.figure(figsize=(20,5))
    a = plt.scatter(data["global_coordinates"][:,0], data["global_coordinates"][:,1], c=data["global_funcvalues"], alpha=.2)
    plt.colorbar(a)
    plt.show()
    print("done!")

    
    #UPDATE GPs
    print("Updating GP agents...")
    update_futures = [] 
    for blockID in data["local_data"]:
        x_data = data["local_data"][blockID]["points"]
        y_data = data["local_data"][blockID]["values"]
        set_args(client, GPs[blockID], {"active": True, "x_data": data["global_coordinates"], "y_data": data["global_funcvalues"], "rbf": rbf, "span": span, "block_mean": np.mean(y_data)})
        update_futures.append(tell(client, x_data, y_data, np.ones(y_data.shape) * noise, GPs[blockID]))
        if iteration_counter in training_at:
            new_hyperparameters = GPs[blockID].result().hyperparameters.copy()
            new_hyperparameters[0] = np.std(y_data)
            set_hps(client, GPs[blockID], new_hyperparameters)
            
    client.gather(update_futures)
    print("Updating GP agents done!")

    #TRAINING
    if iteration_counter in training_at:
        print("Training...")
        train_futures = []
        for blockID in data["local_data"]: 
            if not GPs[blockID].result().args["active"]: continue
            train_futures.append(train(client, hyperparameter_bounds, GPs[blockID],  max_iter = 100, method = "mcmc"))
        client.gather(train_futures)
        print("Training done!")
    if match_hps:
        for entry in GPs: LL[entry] = GPs[entry].result().log_likelihood()
        maxLL = max(LL, key=lambda k: LL[k])
        for entry in GPs: set_hps(client, GPs[entry], GPs[maxLL].result().hyperparameters.copy())
    print("++++++++++++++++++++++++++++++++++")